In [1]:
import pandas as pd
import numpy as np

In [2]:
# Funciones de evaluación de tu paper
def paper_r2(y_true, y_pred):
    """Correlación de Pearson R"""
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return 0.0
    return np.corrcoef(y_true, y_pred)[0, 1]

def rmse(y_true, y_pred):
    """Root Mean Squared Error"""
    return np.sqrt(np.mean((y_true - y_pred)**2))

In [3]:
def encontrar_columna_prediccion(codigo_proteina, columnas_predichas):
    if f"{codigo_proteina}_prediccion" in columnas_predichas: 
        return f"{codigo_proteina}_prediccion"
    if pd.notna(codigo_proteina) and ';' in str(codigo_proteina):
        for sub_codigo in codigo_proteina.split(';'):
            if f"{sub_codigo}_prediccion" in columnas_predichas:
                return f"{sub_codigo}_prediccion"
    return None

In [7]:
# 1. Cargar datos
df_exp = pd.read_csv('datos_np_unq_rpa_externo.csv')
df_pred = pd.read_csv('datos_np_unq_externo_prediction_opt_lip-np_type.csv')
output_filename = "metricas_por_proteina_individual_opt_lip-np_type.csv"

In [8]:
# =========================================================================
# Asegurate de que los índices coincidan con la fila de cada NP en df_pred
mapeo_np = {'PL1': 4, 'PL2': 5, 'DDL1': 6, 'DDL2': 7} 
# =========================================================================

resultados_proteinas = []

# 2. Iterar por PROTEÍNA (por cada fila de los datos experimentales)
for _, fila_exp in df_exp.iterrows():
    codigo = fila_exp['code']
    nombre_proteina = fila_exp['protein'] # Nombre común para identificarla mejor
    col_pred = encontrar_columna_prediccion(codigo, df_pred.columns)
    
    if col_pred is not None:
        y_true = []
        y_pred = []
        
        # Juntar los 4 puntos (uno por cada NP)
        for nombre_np, idx in mapeo_np.items():
            val_real = fila_exp[nombre_np]
            val_pred = df_pred.loc[idx, col_pred]
            
            if pd.notna(val_real) and pd.notna(val_pred):
                y_true.append(val_real)
                y_pred.append(val_pred)
                
        # Calcular solo si tenemos los 4 puntos completos para evitar errores
        if len(y_true) == 4:
            if np.std(y_true) == 0 or np.std(y_pred) == 0:
                r = 0.0
            else:
                r = np.corrcoef(y_true, y_pred)[0, 1]
                
            r2 = r ** 2
            rmse = np.sqrt(np.mean((np.array(y_true) - np.array(y_pred))**2))
            
            resultados_proteinas.append({
                'Código': codigo,
                'Nombre': nombre_proteina,
                'Pearson R': r,
                'Pearson R²': r2,
                'RMSE': rmse
            })

# 3. Mostrar y guardar
df_res_proteinas = pd.DataFrame(resultados_proteinas)

# Ordenamos por R2 para ver las que el modelo predijo mejor
df_res_proteinas = df_res_proteinas.sort_values(by='Pearson R²', ascending=False).reset_index(drop=True)

print("============= RESULTADOS POR PROTEÍNA INDIVIDUAL (Top 15) =============")
display(df_res_proteinas.head(15))

# Exportar para que se lo puedas adjuntar a tu colega
df_res_proteinas.to_csv(output_filename, index=False)
print(f"\nSe exportó la lista completa en {output_filename}")

============= RESULTADOS POR PROTEÍNA INDIVIDUAL (Top 15) =============


,Código,Nombre,Pearson R,Pearson R²,RMSE
0,P04406,GAPDH,-0.989715,0.979537,0.188425
1,P06733,ENO1,-0.987612,0.975377,0.138264
2,P11142,HSPA8,-0.982346,0.965003,0.220649
3,P14618,PKM,-0.973469,0.947642,0.152502
4,P00747,PLG,0.969741,0.940397,0.107945
5,P02647,APOA1,-0.961831,0.925119,0.904595
6,P0C0L5,C4B,-0.959752,0.921123,0.635809
7,P27169,PON1,0.954249,0.910591,0.328004
8,P10909,CLU,0.942277,0.887887,0.534668
9,P02768,ALB,-0.937991,0.879826,3.130881



Se exportó la lista completa en metricas_por_proteina_individual_lip-np_type.csv
